# When Did Congress Start Talking About China?

In this notebook you'll dig through 30 years of congressional floor
speeches (1994–2024) and watch when Congress
started talking about China, what words changed around it, and who drove the surge.

Almost everything is pre-written, and in the marked spots
you'll edit a line of code (a word, a number, or a list) and see what changes.
---

### Step 1

Watch the intro video:

This notebook is your personal copy and your edits save
automatically. If it ever gets hopelessly tangled, **Ctrl+Z / Cmd+Z** inside a cell
fixes almost everything but feel free to message/email Heagen for a new one or any questions if needed.

### How it works

- Cells run top to bottom: click a cell, press **Shift+Enter** (or ▶).
- **JUST RUN** cells: run, don't edit.
- **EDIT** cells: change the line marked `# EDIT`.

## Setup

**JUST RUN:** loads our tools and the three datasets.

In [ ]:
# ▶️ JUST RUN — imports and data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/heagenb03/twl-hs-interns-notebooks/main/data/headlines_real.csv"
mentions   = pd.read_csv(f"{BASE}/congress_mentions.csv")
companions = pd.read_csv(f"{BASE}/congress_companions.csv")
senators   = pd.read_csv(f"{BASE}/congress_senators.csv")

print(f"Loaded {len(mentions)} year-term counts, {len(companions)} companion words, "
      f"{len(senators)} senator-year counts.")
mentions.head()

> **What you should see:** a line saying `Loaded 280 year-term counts, 40 companion
> words, 106 senator-year counts.` and a small table with columns `year`, `term`,
> `mentions`.

Each row of `mentions` answers one question: in year X, how many times did a floor speech mention term Y?

So, which terms can we look up? **JUST RUN:**

In [ ]:
sorted(mentions["term"].unique())

> **What you should see:** a list of 8 terms:
> `['china', 'climate', 'japan', 'russia', 'taiwan', 'tariff', 'terrorism', 'trade']`

Setup cell for plotting **JUST RUN:**

In [ ]:
def plot_terms(terms, start_year, end_year):
    """Plot yearly mention counts for one or more terms."""
    if isinstance(terms, str):
        terms = [terms]  # forgives a bare string instead of a list
    start_year, end_year = int(start_year), int(end_year)
    fig, ax = plt.subplots(figsize=(10, 5))
    for term in terms:
        d = mentions[(mentions.term == term.lower()) &
                     (mentions.year >= start_year) & (mentions.year <= end_year)]
        if len(d) == 0:
            print(f"('{term}' isn't in this dataset. Available: {sorted(mentions.term.unique())})")
            continue
        ax.plot(d.year, d.mentions, marker="o", markersize=3, label=term)
        print(f"'{term}': {d.mentions.iloc[0]} mentions in {d.year.iloc[0]} → "
              f"{d.mentions.iloc[-1]} in {d.year.iloc[-1]} "
              f"(peak: {d.mentions.max()} in {d.loc[d.mentions.idxmax(), 'year']})")
    ax.set_xlabel("year"); ax.set_ylabel("floor-speech mentions per year")
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("How often Congress says it")
    plt.tight_layout(); plt.show()

def compare_eras(era_a, era_b):
    """Side-by-side: which words appear next to 'China' in two different decades."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
    for ax, era in zip(axes, [era_a, era_b]):
        d = companions[companions.era == era].sort_values("count")
        if len(d) == 0:
            print(f"(No era called '{era}'. Options: {sorted(companions.era.unique())})")
            plt.close(fig); return
        ax.barh(d.word, d["count"])
        ax.set_title(f'Words next to "China" in the {era}')
        print(f"{era} top three: {', '.join(d.sort_values('count', ascending=False).word.head(3))}")
    plt.tight_layout(); plt.show()

def plot_senator(name):
    """One senator's China mentions per year vs. the Senate average."""
    d = senators[senators.senator.str.lower() == name.lower()]
    if len(d) == 0:
        print(f"('{name}' isn't in this dataset. Options: {sorted(senators.senator.unique())})")
        return
    avg = senators[senators.senator == "Senate average"]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(d.year, d.mentions, marker="o", label=d.senator.iloc[0])
    ax.plot(avg.year, avg.mentions, linestyle="--", color="gray", label="Senate average")
    ax.set_xlabel("year"); ax.set_ylabel('floor mentions of "China"')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    peak = d.loc[d.mentions.idxmax()]
    print(f"{d.senator.iloc[0]} peaked at {peak.mentions} mentions in {peak.year}.")

def leaderboard(year):
    """Who said 'China' most in a given year?"""
    year = int(year)
    d = senators[(senators.year == year) & (senators.senator != "Senate average")]
    if len(d) == 0:
        print(f"(No data for {year}. This table covers {senators.year.min()}–{senators.year.max()}.)")
        return
    return d.sort_values("mentions", ascending=False)[["senator", "mentions"]].reset_index(drop=True)

def plot_ratio(term_a, term_b):
    """Stretch helper: the ratio of two terms' mentions over time."""
    a = mentions[mentions.term == term_a.lower()].set_index("year").mentions
    b = mentions[mentions.term == term_b.lower()].set_index("year").mentions
    ratio = (a / b).dropna()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ratio.index, ratio.values, marker="o", markersize=3)
    ax.axhline(1, color="gray", linestyle="--")
    ax.set_ylabel(f'"{term_a}" mentions ÷ "{term_b}" mentions')
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    # Mirrors durable_crossover() in scripts/tests/test_plot_ratio_crossover.py - keep both in sync.
    crossed = ratio > 1
    if not crossed.iloc[-1]:
        print(f'"{term_a}" has not passed "{term_b}" for good in this window.')
    else:
        below = crossed[~crossed]
        year = (ratio.index[0] if below.empty
                else ratio.index[ratio.index.get_loc(below.index[-1]) + 1])
        print(f'"{term_a}" passed "{term_b}" for good in {year}.')

print("Done")

---
## Frequency

One line answers the whole notebook's title question. Let's draw it.

**EDIT:** run the cell as-is first. Then change the terms in the list and
the year range, and re-run. Each run is one experiment.

In [ ]:
terms_to_plot = ["china"]      # <-- EDIT this list (try adding "terrorism" or "taiwan")
start_year    = 2010           # <-- EDIT
end_year      = 2024           # <-- EDIT

plot_terms(terms_to_plot, start_year, end_year)

> **What you should see** (with the starting values): a line chart with one rising
> line, and a printed summary like
> `'china': 210 mentions in 1990 → 1048 in 2024 (peak: ...)`
> The line is mostly flat-ish for years, has a visible bump around **2000**, and
> then climbs steeply after **2017**.

### Things to try

1. **The bump around 2000.** Zoom in: set the years to 1996–2004 as something specific happened in Congress in 1999–2000 that made everyone talk about China. Why?

2. **Add `"japan"` to the list** and look at the early 1990s. Japan was the China of the '90s because of trade. So will the China line ever do what the Japan line did in the future?

3. **Add `"russia"`.** Its spikes are more sharp and event-shaped (2014, 2022). What does that difference tell you about how Congress sees China v.s. Russia?

One thing to note is the curve tells you Congress is talking more about China. It cannot tell you whether they're saying the same things more frequently or saying actually something new. How could we tell?

---
## Words around a word

For each decade, we counted which words most often appear in the **same sentence** as "China" in floor speeches.

**EDIT** pick two eras to compare. Options: `"1990s"`, `"2000s"`, `"2010s"`, `"2020s"`.

In [ ]:
era_one = "2010s"   # <-- EDIT
era_two = "2020s"   # <-- EDIT

compare_eras(era_one, era_two)

> **What you should see** (with the starting eras): two horizontal bar charts side by
> side, plus printed lines of the top three terms

### Things to try

1. Walk decade by decade: 1990s→2000s, 2000s→2010s, 2010s→2020s. How would you describe the transition from decade to decade?

2. Track single words across the decades. "Taiwan" appears in the 1990s, then disappears, then comes back. Why would that be?

3. Note that a human never read these speeches. Instead, A program
   counted word co-occurrences, and we can interpret a historical narrative. This is what we can describe as data science when aimed towards any subject.

---
## Who's Driving It?

Who are the individual senators contrubuting to it?

**JUST RUN:**

In [ ]:
#JUST RUN: who said "China" most in 2024?
leaderboard(2024)

> **What you should see:** a ranked table of senators. A few names sit far above
> the rest.

Now pick one and look at their whole trajectory.

**Edit:** available names are in the leaderboard above.

In [ ]:
senator_name = "Marco Rubio"   # <-- EDIT (try "Tom Cotton", "Josh Hawley", "Chuck Schumer")

plot_senator(senator_name)

> **What you should see** (with the starting name): a line for the senator, a gray
> dashed line for the Senate average, and a printed peak like
> `Marco Rubio peaked at ... mentions in 20...`. The senator's line pulls away from
> the average sharply after ~2018.

### Things to try

1. Compare **Tom Cotton** and **Chuck Schumer**. Both talk about China a lot, but one has been steadily elevated since the 2010s and one exploded after 2018. Which pattern suggests a long-held issue, and which suggests responding to a moment?
2. **Josh Hawley** only enters in 2019 (he joined the Senate that year), and enters high. What does it mean that new senators now arrive already talking about China?
3. Run and type in `leaderboard(2012)` in a new cell (Insert → Code cell) and compare it to
   `leaderboard(2024)`. How much of the top five is the same people getting louder vs. new people showing up?

---
## The Crossover

When exactly did China overtake Japan in congressional attention?

**EDIT:**

In [ ]:
term_a = "china"   # <-- EDIT
term_b = "japan"   # <-- EDIT

plot_ratio(term_a, term_b)

> **What you should see:** a line crossing the gray dashed line (ratio = 1), and a
> printed year when `"china"` first overtook `"japan"`.

Try your own ratio. `"china"` vs `"russia"`: which rival owns congressional
attention, and does 2022 change it? What does your own ratio reveal?

---
## Recap

With edited strings in Python in a Jupyter Notebook:

- plotted years of congressional attention
- watched the vocabulary around "China" shift from *trade, WTO*
  to *CCP, threat, genocide*
- and broke the aggregate apart to see which individual senators drive it.

How can we use mass data analysis to determine the past and future? How do you see the future changing as more and more data gets collected everday? The better or for the worst?

Will data tell something a historian couldn't? Does this shape your understanding of data as a whole?

Lastly any thoughts, ideas, concerns on this assignment as a whole?